# Train Arm A (transformer_standard) on Kaggle - full pipeline test

See `plans/PLAN.md` (question 1.3, Arm A), `phases/phase-2-small-train.md`.

Runs the ACTUAL repo code (`vislm/train.py`, `vislm/backbones/`, the real
`experiments/pillar1_patch_encoder/configs/1_3_arm_A_bpe.yaml`) via the
`nguyennn263/vislm-research-code` dataset attached to this kernel (mounted at
`/kaggle/input/datasets/nguyennn263/vislm-research-code` - note the extra `datasets/<owner>/` segment Kaggle adds to the usual
`/kaggle/input/<slug>/` path) - not copy-pasted into the notebook. Config overrides
(`vislm/args.py`) scale it down to a Kaggle-sized debug run; the same config + same code
run the full job later on the RTX 24GB machine by passing different overrides.

Kaggle's free GPU is a Tesla P100 (compute capability sm_60), which torch >=2.8 no
longer supports (dropped with the cu128 builds). Cell 2 detects this and pins
`torch==2.7.1+cu126` (last version with Pascal support) only when a P100 is assigned -
if Kaggle gives a newer GPU (T4/A100), the pre-installed newer torch is left alone.
`vislm/train.py`'s `pick_device()` is still a safety net if this ever misses a case.


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
%env PYTHONPATH=/kaggle/input/datasets/nguyennn263/vislm-research-code
!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python /kaggle/input/datasets/nguyennn263/vislm-research-code/setup/download_prepare_data.py \
  --target-gb 0.05 --out-dir /kaggle/working/data/prepared/fineweb2_vi


In [ ]:
!python -m vislm.train /kaggle/input/datasets/nguyennn263/vislm-research-code/pillar1_configs/1_3_arm_A_bpe.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_A_debug \
  train.max_steps=300


In [ ]:
import json

losses = []
with open("/kaggle/working/runs/arm_A_debug/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            losses.append(row["loss"])

summary = {
    "n_steps": len(losses),
    "first_loss": losses[0],
    "last_loss": losses[-1],
    "min_loss": min(losses),
}
print(summary)

with open("/kaggle/working/metrics_train_arm_a_debug.jsonl", "w") as f:
    f.write(json.dumps({"section": "train_arm_a_debug", "results": summary}) + "\n")
